<a href="https://colab.research.google.com/github/jetsonmom/6.23_automobility_lesson/blob/main/%ED%85%90%EC%84%9Crt%EC%99%80_%ED%8C%8C%EC%9D%B4%ED%86%A0%EC%B9%98_%EB%B9%84%EA%B5%90_ADAS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

# 디스크 공간 확인
!df -h

# CUDA 환경 확인
!nvidia-smi

# GPU 정보 확인
!nvidia-ml-py3 || pip install nvidia-ml-py3

In [ ]:
# 빠른 공간 확인
import shutil
total, used, free = shutil.disk_usage('/')
print(f"💾 디스크 여유공간: {free // (1024**3):.1f}GB")

In [ ]:
# 기본 패키지 설치
!pip install opencv-python
!pip install numpy
!pip install matplotlib
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121
!pip install ultralytics

In [ ]:
# 패키지 임포트 테스트
try:
    import cv2
    import numpy as np
    import matplotlib.pyplot as plt
    print("✅ 기본 패키지 로드 성공")
except ImportError as e:
    print(f"❌ 패키지 로드 실패: {e}")

# CUDA 확인
try:
    import torch
    print(f"✅ PyTorch: {torch.__version__}")
    print(f"✅ CUDA 사용 가능: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"✅ GPU: {torch.cuda.get_device_name(0)}")
except ImportError:
    print("❌ PyTorch 설치 필요")

In [ ]:
import os

# workspace 이미지 확인
image_files = ['1.jpg', '2.jpg', '3.jpg']
found_images = []

print("📁 이미지 파일 확인...")
for img_file in image_files:
    full_path = f'/workspace/{img_file}'
    if os.path.exists(full_path):
        try:
            img = cv2.imread(full_path)
            if img is not None:
                h, w, c = img.shape
                file_size = os.path.getsize(full_path) / 1024  # KB
                print(f"✅ {img_file}: {w}x{h}, {file_size:.1f}KB")
                found_images.append(full_path)
            else:
                print(f"❌ {img_file}: 이미지 읽기 실패")
        except Exception as e:
            print(f"❌ {img_file}: 오류 - {e}")
    else:
        print(f"❌ {img_file}: 파일 없음")

print(f"\n📊 결과: {len(found_images)}개 이미지 발견")

In [ ]:
# YOLO 모델 로드 (처음에는 다운로드 시간이 걸릴 수 있습니다)
from ultralytics import YOLO

print("📦 YOLO 모델 로딩...")
model = YOLO('yolov8n.pt')  # nano 버전 (가장 빠름)
print("✅ YOLO 모델 로드 완료")

# GPU 사용 설정
if torch.cuda.is_available():
    model.to('cuda')
    print("✅ GPU로 모델 이동 완료")

In [ ]:
import time
# 첫 번째 이미지로 간단 테스트
if found_images:
    test_image = found_images[0]
    print(f"🔍 테스트 이미지: {os.path.basename(test_image)}")

    # YOLO 추론
    start_time = time.time()
    results = model(test_image, conf=0.5)
    inference_time = time.time() - start_time

    print(f"⏱️ 추론 시간: {inference_time:.3f}초")

    # 결과 확인
    for result in results:
        boxes = result.boxes
        if boxes is not None:
            print(f"🎯 감지된 객체: {len(boxes)}개")

            # 각 객체 정보 출력
            for i, box in enumerate(boxes):
                class_id = int(box.cls[0])
                class_name = result.names[class_id]
                confidence = box.conf[0].cpu().numpy()
                print(f"  {i+1}. {class_name}: {confidence:.2f}")
        else:
            print("❌ 감지된 객체 없음")
else:
    print("❌ 테스트할 이미지가 없습니다")

In [ ]:
# Complete ADAS System - 3 Images Comparison with Lane Detection
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
import torch
from ultralytics import YOLO
import os

class ComprehensiveADAS:
    """Complete ADAS System with Object Detection and Lane Detection"""

    def __init__(self):
        print("🚗 Initializing Comprehensive ADAS System...")
        self.model = None
        self.load_model()

    def load_model(self):
        """Load YOLO model"""
        try:
            print("📦 Loading YOLO model...")
            self.model = YOLO('yolov8n.pt')

            if torch.cuda.is_available():
                self.model.to('cuda')
                print("✅ Model loaded on GPU")
            else:
                print("✅ Model loaded on CPU")

        except Exception as e:
            print(f"❌ Model loading failed: {e}")

    def detect_objects(self, image_path):
        """Object detection with performance measurement"""
        print(f"🔍 Object Detection: {os.path.basename(image_path)}")

        # Multiple inference for accurate FPS (20 iterations like PyTorch)
        times = []
        print(f"  Running 20 iterations for accurate measurement...")
        for i in range(20):
            start_time = time.time()
            results = self.model(image_path, conf=0.5, verbose=False)
            inference_time = time.time() - start_time
            times.append(inference_time)
            if (i + 1) % 5 == 0:
                print(f"    Iteration {i+1}/20: {inference_time:.4f}s ({1.0/inference_time:.1f} FPS)")

        avg_time = np.mean(times)
        std_time = np.std(times)
        min_time = np.min(times)
        max_time = np.max(times)
        fps = 1.0 / avg_time
        max_fps = 1.0 / min_time
        min_fps = 1.0 / max_time

        # Parse results
        result = results[0]
        detections = []
        adas_objects = ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck', 'traffic light', 'stop sign']

        if result.boxes is not None:
            for box in result.boxes:
                class_id = int(box.cls[0])
                class_name = result.names[class_id]
                confidence = float(box.conf[0])
                x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()

                if class_name in adas_objects:
                    detections.append({
                        'class': class_name,
                        'confidence': confidence,
                        'bbox': [int(x1), int(y1), int(x2), int(y2)],
                        'center': [(x1 + x2) / 2, (y1 + y2) / 2]
                    })

        print(f"  ⚡ Average FPS: {fps:.2f} (±{std_time*fps*fps:.2f})")
        print(f"  📊 FPS Range: {min_fps:.2f} - {max_fps:.2f}")
        print(f"  ⏱️ Time: {avg_time:.4f}s (±{std_time:.4f}s)")
        print(f"  🎯 ADAS Objects: {len(detections)}")

        return {
            'detections': detections,
            'fps': fps,
            'fps_std': std_time * fps * fps,
            'fps_min': min_fps,
            'fps_max': max_fps,
            'inference_time': avg_time,
            'time_std': std_time,
            'all_times': times,
            'yolo_result': result
        }

    def detect_lanes(self, image_path):
        """Advanced lane detection"""
        print(f"🛣️ Lane Detection: {os.path.basename(image_path)}")

        # Load image
        image = cv2.imread(image_path)
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        height, width = gray.shape

        # Preprocessing
        blur = cv2.GaussianBlur(gray, (5, 5), 0)

        # Edge detection
        edges = cv2.Canny(blur, 50, 150, apertureSize=3)

        # Define ROI (Region of Interest)
        roi_vertices = np.array([[
            (int(width * 0.1), height),
            (int(width * 0.45), int(height * 0.6)),
            (int(width * 0.55), int(height * 0.6)),
            (int(width * 0.9), height)
        ]], dtype=np.int32)

        # Create mask
        mask = np.zeros_like(edges)
        cv2.fillPoly(mask, roi_vertices, 255)
        masked_edges = cv2.bitwise_and(edges, mask)

        # Hough Line Transform
        lines = cv2.HoughLinesP(
            masked_edges,
            rho=1,
            theta=np.pi/180,
            threshold=50,
            minLineLength=100,
            maxLineGap=50
        )

        # Classify lines (left/right lanes)
        left_lines = []
        right_lines = []

        if lines is not None:
            for line in lines:
                x1, y1, x2, y2 = line[0]
                if x2 != x1:  # Avoid division by zero
                    slope = (y2 - y1) / (x2 - x1)
                    if abs(slope) > 0.3:  # Filter by slope
                        if slope < 0:  # Left lane
                            left_lines.append([x1, y1, x2, y2])
                        else:  # Right lane
                            right_lines.append([x1, y1, x2, y2])

        print(f"  📊 Left lanes: {len(left_lines)}, Right lanes: {len(right_lines)}")

        return {
            'left_lines': left_lines,
            'right_lines': right_lines,
            'roi_vertices': roi_vertices[0],
            'total_lines': len(left_lines) + len(right_lines)
        }

    def collision_risk_analysis(self, detections, image_shape):
        """Collision risk assessment"""
        height, width = image_shape[:2]

        # Define danger zones
        critical_zone = {
            'x1': width * 0.3, 'y1': height * 0.7,
            'x2': width * 0.7, 'y2': height
        }

        warning_zone = {
            'x1': width * 0.2, 'y1': height * 0.5,
            'x2': width * 0.8, 'y2': height
        }

        warnings = []

        for detection in detections:
            x1, y1, x2, y2 = detection['bbox']
            center_x, center_y = detection['center']

            # Check critical zone
            if (x1 < critical_zone['x2'] and x2 > critical_zone['x1'] and
                y1 < critical_zone['y2'] and y2 > critical_zone['y1']):

                risk_level = 'CRITICAL'
                risk_color = 'red'

            # Check warning zone
            elif (x1 < warning_zone['x2'] and x2 > warning_zone['x1'] and
                  y1 < warning_zone['y2'] and y2 > warning_zone['y1']):

                risk_level = 'WARNING'
                risk_color = 'orange'
            else:
                continue

            # Distance estimation based on object size
            obj_area = (x2 - x1) * (y2 - y1)
            screen_area = width * height
            size_ratio = obj_area / screen_area

            if size_ratio > 0.1:
                distance = 'CLOSE'
            elif size_ratio > 0.05:
                distance = 'MEDIUM'
            else:
                distance = 'FAR'

            warnings.append({
                'object': detection['class'],
                'confidence': detection['confidence'],
                'risk_level': risk_level,
                'distance': distance,
                'position': [center_x, center_y]
            })

        return warnings

    def process_single_image(self, image_path):
        """Complete processing of single image"""
        print(f"\n🚀 Processing: {os.path.basename(image_path)}")
        print("=" * 50)

        # Object detection
        obj_results = self.detect_objects(image_path)

        # Lane detection
        lane_results = self.detect_lanes(image_path)

        # Load image for collision analysis
        image = cv2.imread(image_path)

        # Collision risk analysis
        warnings = self.collision_risk_analysis(obj_results['detections'], image.shape)

        # Summary
        print(f"📊 Summary:")
        print(f"  Objects: {len(obj_results['detections'])}")
        print(f"  Lanes: {lane_results['total_lines']}")
        print(f"  Warnings: {len(warnings)}")
        print(f"  FPS: {obj_results['fps']:.1f}")

        return {
            'image_path': image_path,
            'objects': obj_results,
            'lanes': lane_results,
            'warnings': warnings,
            'original_image': image
        }

    def visualize_results(self, result):
        """Visualize complete ADAS results"""
        image_path = result['image_path']
        objects = result['objects']
        lanes = result['lanes']
        warnings = result['warnings']

        # Load original image
        original = cv2.imread(image_path)
        original_rgb = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)

        # Create result image
        result_image = original_rgb.copy()
        height, width = original_rgb.shape[:2]

        # Draw object detections
        for detection in objects['detections']:
            x1, y1, x2, y2 = detection['bbox']
            confidence = detection['confidence']
            class_name = detection['class']

            # Color coding by object type
            colors = {
                'person': (255, 0, 0),      # Red
                'car': (0, 255, 0),         # Green
                'truck': (0, 0, 255),       # Blue
                'bus': (255, 255, 0),       # Yellow
                'bicycle': (255, 0, 255),   # Magenta
                'motorcycle': (0, 255, 255), # Cyan
                'traffic light': (255, 165, 0), # Orange
                'stop sign': (128, 0, 128)  # Purple
            }
            color = colors.get(class_name, (128, 128, 128))

            # Draw bounding box
            cv2.rectangle(result_image, (x1, y1), (x2, y2), color, 2)

            # Draw label
            label = f"{class_name}: {confidence:.2f}"
            cv2.putText(result_image, label, (x1, y1-10),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1)

        # Draw lanes
        # Left lanes (blue)
        for line in lanes['left_lines']:
            x1, y1, x2, y2 = line
            cv2.line(result_image, (x1, y1), (x2, y2), (255, 0, 0), 3)

        # Right lanes (yellow)
        for line in lanes['right_lines']:
            x1, y1, x2, y2 = line
            cv2.line(result_image, (x1, y1), (x2, y2), (0, 255, 255), 3)

        # Draw ROI
        roi_pts = lanes['roi_vertices'].reshape((-1, 1, 2))
        cv2.polylines(result_image, [roi_pts], True, (255, 255, 255), 1)

        # Draw danger zones
        # Critical zone (red overlay)
        overlay = result_image.copy()
        cv2.rectangle(overlay,
                     (int(width*0.3), int(height*0.7)),
                     (int(width*0.7), height),
                     (255, 0, 0), -1)
        result_image = cv2.addWeighted(result_image, 0.9, overlay, 0.1, 0)

        # Warning zone (orange overlay)
        cv2.rectangle(overlay,
                     (int(width*0.2), int(height*0.5)),
                     (int(width*0.8), height),
                     (255, 165, 0), -1)
        result_image = cv2.addWeighted(result_image, 0.95, overlay, 0.05, 0)

        # Display warnings
        if warnings:
            warning_text = f"⚠️ {len(warnings)} WARNINGS"
            cv2.putText(result_image, warning_text, (50, 50),
                       cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 0, 0), 2)

        # Display performance info
        info_lines = [
            f"FPS: {objects['fps']:.1f}",
            f"Objects: {len(objects['detections'])}",
            f"Lanes: {lanes['total_lines']}",
            f"Warnings: {len(warnings)}"
        ]

        for i, line in enumerate(info_lines):
            cv2.putText(result_image, line, (50, height - 120 + i*25),
                       cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)

        return original_rgb, result_image

def compare_all_images():
    """Compare all 3 images with complete ADAS analysis"""
    print("🚗💨 Starting Complete ADAS Analysis for All Images")
    print("=" * 70)

    # Check available images
    image_files = ['1.jpg', '2.jpg', '3.jpg']
    available_images = []

    for img_file in image_files:
        full_path = f'/workspace/{img_file}'
        if os.path.exists(full_path):
            available_images.append(full_path)
            print(f"✅ Found: {img_file}")
        else:
            print(f"❌ Missing: {img_file}")

    if not available_images:
        print("❌ No images found!")
        return

    # Initialize ADAS system
    adas = ComprehensiveADAS()

    # Process all images
    all_results = []
    total_objects = 0
    total_lanes = 0
    total_warnings = 0
    total_time = 0

    for image_path in available_images:
        result = adas.process_single_image(image_path)
        all_results.append(result)

        total_objects += len(result['objects']['detections'])
        total_lanes += result['lanes']['total_lines']
        total_warnings += len(result['warnings'])
        total_time += result['objects']['inference_time']

    # Visualization
    print(f"\n🖼️ Visualizing Results...")

    fig, axes = plt.subplots(len(available_images), 2, figsize=(20, 6*len(available_images)))
    if len(available_images) == 1:
        axes = axes.reshape(1, -1)

    for i, result in enumerate(all_results):
        original, processed = adas.visualize_results(result)

        # Original image
        axes[i, 0].imshow(original)
        axes[i, 0].set_title(f"Original: {os.path.basename(result['image_path'])}", fontsize=12)
        axes[i, 0].axis('off')

        # Processed image
        axes[i, 1].imshow(processed)
        fps = result['objects']['fps']
        obj_count = len(result['objects']['detections'])
        lane_count = result['lanes']['total_lines']
        warning_count = len(result['warnings'])

        title = f"ADAS Result: {fps:.1f}FPS, {obj_count}obj, {lane_count}lanes, {warning_count}warn"
        axes[i, 1].set_title(title, fontsize=12)
        axes[i, 1].axis('off')

    plt.tight_layout()
    plt.show()

    # Overall statistics
    print(f"\n📊 DETAILED PERFORMANCE STATISTICS (20 iterations each)")
    print("=" * 70)
    print(f"📁 Processed Images: {len(available_images)}")
    print(f"🎯 Total Objects Detected: {total_objects}")
    print(f"🛣️ Total Lane Lines: {total_lanes}")
    print(f"⚠️ Total Warnings: {total_warnings}")
    print(f"⏱️ Total Processing Time: {total_time:.3f}s")
    print(f"📈 Average FPS Across All Images: {len(available_images)/total_time:.2f}")

    # FPS Statistics
    all_fps = [r['objects']['fps'] for r in all_results]
    all_fps_min = [r['objects']['fps_min'] for r in all_results]
    all_fps_max = [r['objects']['fps_max'] for r in all_results]

    print(f"\n⚡ FPS PERFORMANCE SUMMARY:")
    print(f"  Overall Average FPS: {np.mean(all_fps):.2f}")
    print(f"  Overall FPS Range: {np.min(all_fps_min):.2f} - {np.max(all_fps_max):.2f}")
    print(f"  Best Single Image FPS: {np.max(all_fps):.2f}")
    print(f"  Worst Single Image FPS: {np.min(all_fps):.2f}")
    print(f"  FPS Standard Deviation: {np.std(all_fps):.2f}")

    # Detailed breakdown
    print(f"\n📋 DETAILED BREAKDOWN")
    print("-" * 50)

    for i, result in enumerate(all_results):
        img_name = os.path.basename(result['image_path'])
        objects = result['objects']
        lanes = result['lanes']
        warnings = result['warnings']

        print(f"\n{i+1}. {img_name}:")
        print(f"   Average FPS: {objects['fps']:.2f} (±{objects['fps_std']:.2f})")
        print(f"   FPS Range: {objects['fps_min']:.2f} - {objects['fps_max']:.2f}")
        print(f"   Average Time: {objects['inference_time']:.4f}s (±{objects['time_std']:.4f}s)")
        print(f"   Objects: {len(objects['detections'])}")

        # Object breakdown
        obj_types = {}
        for det in objects['detections']:
            obj_type = det['class']
            obj_types[obj_type] = obj_types.get(obj_type, 0) + 1

        for obj_type, count in obj_types.items():
            print(f"     - {obj_type}: {count}")

        print(f"   Lanes: L={len(lanes['left_lines'])}, R={len(lanes['right_lines'])}")
        print(f"   Warnings: {len(warnings)}")

        for warning in warnings:
            print(f"     - {warning['object']}: {warning['risk_level']} ({warning['distance']})")

    # Performance comparison with detailed statistics
    if len(all_results) > 1:
        print(f"\n⚡ DETAILED PERFORMANCE COMPARISON")
        print("-" * 50)

        fps_values = [r['objects']['fps'] for r in all_results]
        fps_mins = [r['objects']['fps_min'] for r in all_results]
        fps_maxs = [r['objects']['fps_max'] for r in all_results]
        obj_counts = [len(r['objects']['detections']) for r in all_results]

        print(f"Fastest Average FPS: {max(fps_values):.2f}")
        print(f"Slowest Average FPS: {min(fps_values):.2f}")
        print(f"Peak FPS (single inference): {max(fps_maxs):.2f}")
        print(f"Lowest FPS (single inference): {min(fps_mins):.2f}")
        print(f"Most Objects Detected: {max(obj_counts)} objects")
        print(f"Least Objects Detected: {min(obj_counts)} objects")

        # Consistency analysis
        fps_consistency = [r['objects']['time_std'] for r in all_results]
        most_consistent_idx = np.argmin(fps_consistency)
        least_consistent_idx = np.argmax(fps_consistency)

        print(f"\nConsistency Analysis:")
        print(f"Most Consistent: {os.path.basename(all_results[most_consistent_idx]['image_path'])} "
              f"(std: {fps_consistency[most_consistent_idx]:.4f}s)")
        print(f"Least Consistent: {os.path.basename(all_results[least_consistent_idx]['image_path'])} "
              f"(std: {fps_consistency[least_consistent_idx]:.4f}s)")

    print(f"\n✅ Complete ADAS Analysis Finished!")

    return all_results

# Execute the complete analysis
if __name__ == "__main__":
    results = compare_all_images()

In [ ]:
# TensorRT Optimized ADAS Performance Comparison
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
import torch
from ultralytics import YOLO
import os

class TensorRTADAS:
    """TensorRT Optimized ADAS System"""

    def __init__(self):
        print("🚀 Loading TensorRT Optimized YOLO model...")
        self.model = None
        self.tensorrt_model = None
        self.load_models()

    def load_models(self):
        """Load both PyTorch and TensorRT models for comparison"""
        # Load original PyTorch model
        self.model = YOLO('yolov8n.pt')
        if torch.cuda.is_available():
            self.model.to('cuda')
            print("✅ PyTorch model loaded on GPU")

        # Export to TensorRT for optimization
        try:
            print("🔄 Exporting to TensorRT...")
            # Export model to TensorRT format
            self.model.export(format='engine', imgsz=640, half=True, device=0)

            # Load TensorRT optimized model
            model_name = str(self.model.ckpt_path or 'yolov8n.pt').replace('.pt', '.engine')
            if os.path.exists(model_name):
                self.tensorrt_model = YOLO(model_name)
                print("✅ TensorRT model loaded successfully")
            else:
                # Try alternative path
                tensorrt_path = 'yolov8n.engine'
                if os.path.exists(tensorrt_path):
                    self.tensorrt_model = YOLO(tensorrt_path)
                    print("✅ TensorRT model loaded successfully")
                else:
                    print("⚠️ TensorRT engine not found, using PyTorch with optimizations...")
                    self.tensorrt_model = self.model
                    self.optimize_pytorch_model()

        except Exception as e:
            print(f"⚠️ TensorRT export failed: {e}")
            print("🔄 Using optimized PyTorch instead...")
            self.tensorrt_model = self.model
            self.optimize_pytorch_model()

    def optimize_pytorch_model(self):
        """Apply PyTorch optimizations when TensorRT is not available"""
        if torch.cuda.is_available():
            # Enable optimizations
            torch.backends.cudnn.benchmark = True
            torch.backends.cudnn.deterministic = False
            print("✅ Applied PyTorch optimizations (CUDNN benchmark enabled)")

        # Keep original model for compatibility
        self.tensorrt_model = self.model

    def benchmark_inference(self, image_path, model_type='tensorrt', iterations=20):
        """Benchmark inference performance"""
        model = self.tensorrt_model if model_type == 'tensorrt' else self.model

        print(f"🏃 Running {model_type.upper()} benchmark ({iterations} iterations)...")

        # Warmup
        for _ in range(3):
            _ = model(image_path, conf=0.5, verbose=False)

        # Actual benchmark
        times = []
        torch.cuda.synchronize() if torch.cuda.is_available() else None

        for i in range(iterations):
            start_time = time.time()
            results = model(image_path, conf=0.5, verbose=False)
            torch.cuda.synchronize() if torch.cuda.is_available() else None
            inference_time = time.time() - start_time
            times.append(inference_time)

            if (i + 1) % 5 == 0:
                current_fps = 1.0 / inference_time
                print(f"    Iteration {i+1}: {inference_time:.4f}s ({current_fps:.1f} FPS)")

        # Calculate statistics
        avg_time = np.mean(times)
        std_time = np.std(times)
        min_time = np.min(times)
        max_time = np.max(times)
        avg_fps = 1.0 / avg_time
        max_fps = 1.0 / min_time
        min_fps = 1.0 / max_time

        # Parse detections
        result = results[0]
        detections = []
        adas_objects = ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck', 'traffic light', 'stop sign']

        if result.boxes is not None:
            for box in result.boxes:
                class_id = int(box.cls[0])
                class_name = result.names[class_id]
                confidence = float(box.conf[0])

                if class_name in adas_objects:
                    detections.append({
                        'class': class_name,
                        'confidence': confidence
                    })

        return {
            'avg_fps': avg_fps,
            'min_fps': min_fps,
            'max_fps': max_fps,
            'avg_time': avg_time,
            'std_time': std_time,
            'fps_std': std_time * avg_fps * avg_fps,
            'all_times': times,
            'objects': len(detections),
            'detections': detections
        }

    def detect_lanes(self, image_path):
        """Simple lane detection"""
        image = cv2.imread(image_path)
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # Optimized preprocessing
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        edges = cv2.Canny(blur, 50, 150)

        # ROI
        height, width = gray.shape
        mask = np.zeros_like(edges)
        polygon = np.array([[
            (width//4, height),
            (width//2 - 50, height//2 + 50),
            (width//2 + 50, height//2 + 50),
            (3*width//4, height)
        ]], np.int32)
        cv2.fillPoly(mask, polygon, 255)
        masked_edges = cv2.bitwise_and(edges, mask)

        # Line detection
        lines = cv2.HoughLinesP(masked_edges, 1, np.pi/180, 50,
                               minLineLength=100, maxLineGap=50)

        return len(lines) if lines is not None else 0

    def compare_performance(self, image_path):
        """Compare PyTorch vs TensorRT performance"""
        print(f"\n🔥 Performance Comparison: {os.path.basename(image_path)}")
        print("=" * 60)

        # PyTorch benchmark
        pytorch_results = self.benchmark_inference(image_path, 'pytorch', 20)

        # TensorRT benchmark
        tensorrt_results = self.benchmark_inference(image_path, 'tensorrt', 20)

        # Lane detection (same for both)
        lanes = self.detect_lanes(image_path)

        # Calculate improvements
        fps_improvement = (tensorrt_results['avg_fps'] / pytorch_results['avg_fps'] - 1) * 100
        time_improvement = (pytorch_results['avg_time'] / tensorrt_results['avg_time'] - 1) * 100

        print(f"\n📊 COMPARISON RESULTS:")
        print(f"  PyTorch FPS: {pytorch_results['avg_fps']:.2f} (±{pytorch_results['fps_std']:.2f})")
        print(f"  TensorRT FPS: {tensorrt_results['avg_fps']:.2f} (±{tensorrt_results['fps_std']:.2f})")
        print(f"  🚀 FPS Improvement: {fps_improvement:.1f}%")
        print(f"  ⚡ Time Reduction: {time_improvement:.1f}%")
        print(f"  🎯 Objects: PT={pytorch_results['objects']}, RT={tensorrt_results['objects']}")
        print(f"  🛣️ Lanes: {lanes}")

        return {
            'image_name': os.path.basename(image_path),
            'pytorch': pytorch_results,
            'tensorrt': tensorrt_results,
            'lanes': lanes,
            'fps_improvement': fps_improvement,
            'time_improvement': time_improvement
        }

def create_tensorrt_comparison_charts():
    """Create TensorRT vs PyTorch comparison charts"""
    print("🚀 Creating TensorRT vs PyTorch Comparison Charts...")

    # Check available images
    image_files = ['1.jpg', '2.jpg', '3.jpg']
    available_images = []

    for img_file in image_files:
        full_path = f'/workspace/{img_file}'
        if os.path.exists(full_path):
            available_images.append(full_path)

    if not available_images:
        print("❌ No images found!")
        return

    # Initialize TensorRT ADAS
    adas = TensorRTADAS()

    # Compare all images
    results = []
    for image_path in available_images:
        result = adas.compare_performance(image_path)
        results.append(result)

    # Create comparison charts
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('TensorRT vs PyTorch ADAS Performance Comparison', fontsize=16, fontweight='bold')

    # Extract data
    image_names = [r['image_name'] for r in results]
    pytorch_fps = [r['pytorch']['avg_fps'] for r in results]
    tensorrt_fps = [r['tensorrt']['avg_fps'] for r in results]
    fps_improvements = [r['fps_improvement'] for r in results]
    pytorch_objects = [r['pytorch']['objects'] for r in results]
    tensorrt_objects = [r['tensorrt']['objects'] for r in results]
    lanes = [r['lanes'] for r in results]

    # Colors
    pytorch_color = '#FF6B6B'
    tensorrt_color = '#4ECDC4'
    improvement_color = '#45B7D1'

    # 1. FPS Comparison
    x = np.arange(len(image_names))
    width = 0.35

    bars1 = axes[0, 0].bar(x - width/2, pytorch_fps, width, label='PyTorch',
                          color=pytorch_color, alpha=0.8)
    bars2 = axes[0, 0].bar(x + width/2, tensorrt_fps, width, label='TensorRT',
                          color=tensorrt_color, alpha=0.8)

    axes[0, 0].set_title('FPS Performance Comparison', fontweight='bold')
    axes[0, 0].set_ylabel('Frames Per Second')
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(image_names)
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Add value labels
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            axes[0, 0].text(bar.get_x() + bar.get_width()/2., height + 0.5,
                           f'{height:.1f}', ha='center', va='bottom', fontweight='bold')

    # 2. Performance Improvement
    bars = axes[0, 1].bar(image_names, fps_improvements, color=improvement_color, alpha=0.8)
    axes[0, 1].set_title('FPS Improvement (%)', fontweight='bold')
    axes[0, 1].set_ylabel('Improvement Percentage')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].axhline(y=0, color='black', linestyle='-', alpha=0.5)

    # Add value labels
    for bar in bars:
        height = bar.get_height()
        axes[0, 1].text(bar.get_x() + bar.get_width()/2., height + 1,
                       f'{height:.1f}%', ha='center', va='bottom', fontweight='bold')

    # 3. Object Detection Accuracy
    bars1 = axes[0, 2].bar(x - width/2, pytorch_objects, width, label='PyTorch',
                          color=pytorch_color, alpha=0.8)
    bars2 = axes[0, 2].bar(x + width/2, tensorrt_objects, width, label='TensorRT',
                          color=tensorrt_color, alpha=0.8)

    axes[0, 2].set_title('Objects Detected', fontweight='bold')
    axes[0, 2].set_ylabel('Number of Objects')
    axes[0, 2].set_xticks(x)
    axes[0, 2].set_xticklabels(image_names)
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)

    # Add value labels
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            axes[0, 2].text(bar.get_x() + bar.get_width()/2., height + 0.1,
                           f'{int(height)}', ha='center', va='bottom', fontweight='bold')

    # 4. Processing Time Comparison
    pytorch_times = [r['pytorch']['avg_time'] * 1000 for r in results]  # Convert to ms
    tensorrt_times = [r['tensorrt']['avg_time'] * 1000 for r in results]

    bars1 = axes[1, 0].bar(x - width/2, pytorch_times, width, label='PyTorch',
                          color=pytorch_color, alpha=0.8)
    bars2 = axes[1, 0].bar(x + width/2, tensorrt_times, width, label='TensorRT',
                          color=tensorrt_color, alpha=0.8)

    axes[1, 0].set_title('Processing Time Comparison', fontweight='bold')
    axes[1, 0].set_ylabel('Time (milliseconds)')
    axes[1, 0].set_xticks(x)
    axes[1, 0].set_xticklabels(image_names)
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Add value labels
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            axes[1, 0].text(bar.get_x() + bar.get_width()/2., height + 1,
                           f'{height:.1f}', ha='center', va='bottom', fontweight='bold')

    # 5. Performance Summary
    axes[1, 1].axis('off')

    avg_pytorch_fps = np.mean(pytorch_fps)
    avg_tensorrt_fps = np.mean(tensorrt_fps)
    avg_improvement = np.mean(fps_improvements)

    summary_text = f"""
TENSORRT OPTIMIZATION RESULTS
{'='*35}

Average Performance:
• PyTorch FPS: {avg_pytorch_fps:.1f}
• TensorRT FPS: {avg_tensorrt_fps:.1f}
• Average Improvement: {avg_improvement:.1f}%

Best Performance:
• Highest FPS: {max(tensorrt_fps):.1f} ({image_names[np.argmax(tensorrt_fps)]})
• Best Improvement: {max(fps_improvements):.1f}% ({image_names[np.argmax(fps_improvements)]})

Detection Accuracy:
• PyTorch Objects: {sum(pytorch_objects)}
• TensorRT Objects: {sum(tensorrt_objects)}
• Accuracy Maintained: {'✅' if sum(pytorch_objects) == sum(tensorrt_objects) else '⚠️'}

Lane Detection:
• Total Lanes: {sum(lanes)}
• Avg per Image: {np.mean(lanes):.1f}
"""

    axes[1, 1].text(0.05, 0.95, summary_text, transform=axes[1, 1].transAxes,
                    fontsize=10, verticalalignment='top', fontfamily='monospace',
                    bbox=dict(boxstyle="round,pad=0.5", facecolor="lightgreen", alpha=0.8))

    # 6. Speed vs Accuracy Trade-off
    axes[1, 2].scatter(pytorch_fps, pytorch_objects, s=100, color=pytorch_color,
                      alpha=0.8, label='PyTorch', marker='o')
    axes[1, 2].scatter(tensorrt_fps, tensorrt_objects, s=100, color=tensorrt_color,
                      alpha=0.8, label='TensorRT', marker='s')

    # Add image labels
    for i, name in enumerate(image_names):
        axes[1, 2].annotate(name, (pytorch_fps[i], pytorch_objects[i]),
                           xytext=(5, 5), textcoords='offset points', fontsize=8)
        axes[1, 2].annotate(name, (tensorrt_fps[i], tensorrt_objects[i]),
                           xytext=(5, -10), textcoords='offset points', fontsize=8)

    axes[1, 2].set_title('Speed vs Accuracy Trade-off', fontweight='bold')
    axes[1, 2].set_xlabel('FPS (Speed)')
    axes[1, 2].set_ylabel('Objects Detected (Accuracy)')
    axes[1, 2].legend()
    axes[1, 2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Print detailed comparison
    print(f"\n🏆 TENSORRT OPTIMIZATION SUMMARY")
    print("="*60)

    for i, result in enumerate(results):
        print(f"\n{i+1}. {result['image_name']}:")
        print(f"   PyTorch:  {result['pytorch']['avg_fps']:.2f} FPS ({result['pytorch']['avg_time']*1000:.1f}ms)")
        print(f"   TensorRT: {result['tensorrt']['avg_fps']:.2f} FPS ({result['tensorrt']['avg_time']*1000:.1f}ms)")
        print(f"   🚀 Improvement: {result['fps_improvement']:.1f}% faster")
        print(f"   🎯 Objects: {result['pytorch']['objects']} → {result['tensorrt']['objects']}")
        print(f"   🛣️ Lanes: {result['lanes']}")

    print(f"\n🎉 OVERALL OPTIMIZATION RESULTS:")
    print(f"   Average FPS improvement: {avg_improvement:.1f}%")
    print(f"   Best single improvement: {max(fps_improvements):.1f}%")
    print(f"   Accuracy maintained: {'Yes' if sum(pytorch_objects) == sum(tensorrt_objects) else 'No'}")

    return results

# Main execution
def main():
    """Main execution for TensorRT comparison"""
    print("🚀💨 Starting TensorRT vs PyTorch ADAS Comparison")
    print("="*60)

    # Create TensorRT comparison charts
    results = create_tensorrt_comparison_charts()

    print("\n✅ TensorRT optimization analysis complete!")
    print("🏆 Performance improvements measured and visualized!")

    return results

if __name__ == "__main__":
    tensorrt_results = main()

In [ ]:
# TensorRT Optimized ADAS Performance Comparison
import cv2
import numpy as np
import matplotlib.pyplot as plt
import time
import torch
from ultralytics import YOLO
import os

class TensorRTADAS:
    """TensorRT Optimized ADAS System"""

    def __init__(self):
        print("🚀 Loading TensorRT Optimized YOLO model...")
        self.model = None
        self.tensorrt_model = None
        self.load_models()

    def load_models(self):
        """Load both PyTorch and TensorRT models for comparison"""
        # Load original PyTorch model
        self.model = YOLO('yolov8n.pt')
        if torch.cuda.is_available():
            self.model.to('cuda')
            print("✅ PyTorch model loaded on GPU")

        # Export to TensorRT for optimization
        try:
            print("🔄 Exporting to TensorRT...")
            # Export model to TensorRT format
            self.model.export(format='engine', imgsz=640, half=True, device=0)

            # Load TensorRT optimized model
            model_name = str(self.model.ckpt_path or 'yolov8n.pt').replace('.pt', '.engine')
            if os.path.exists(model_name):
                self.tensorrt_model = YOLO(model_name)
                print("✅ TensorRT model loaded successfully")
            else:
                # Try alternative path
                tensorrt_path = 'yolov8n.engine'
                if os.path.exists(tensorrt_path):
                    self.tensorrt_model = YOLO(tensorrt_path)
                    print("✅ TensorRT model loaded successfully")
                else:
                    print("⚠️ TensorRT engine not found, using PyTorch with optimizations...")
                    self.tensorrt_model = self.model
                    self.optimize_pytorch_model()

        except Exception as e:
            print(f"⚠️ TensorRT export failed: {e}")
            print("🔄 Using optimized PyTorch instead...")
            self.tensorrt_model = self.model
            self.optimize_pytorch_model()

    def optimize_pytorch_model(self):
        """Apply PyTorch optimizations when TensorRT is not available"""
        if torch.cuda.is_available():
            # Enable optimizations
            torch.backends.cudnn.benchmark = True
            torch.backends.cudnn.deterministic = False
            print("✅ Applied PyTorch optimizations (CUDNN benchmark enabled)")

        # Keep original model for compatibility
        self.tensorrt_model = self.model

    def benchmark_inference(self, image_path, model_type='tensorrt', iterations=20):
        """Benchmark inference performance"""
        model = self.tensorrt_model if model_type == 'tensorrt' else self.model

        print(f"🏃 Running {model_type.upper()} benchmark ({iterations} iterations)...")

        # Warmup
        for _ in range(3):
            _ = model(image_path, conf=0.5, verbose=False)

        # Actual benchmark
        times = []
        torch.cuda.synchronize() if torch.cuda.is_available() else None

        for i in range(iterations):
            start_time = time.time()
            results = model(image_path, conf=0.5, verbose=False)
            torch.cuda.synchronize() if torch.cuda.is_available() else None
            inference_time = time.time() - start_time
            times.append(inference_time)

            if (i + 1) % 5 == 0:
                current_fps = 1.0 / inference_time
                print(f"    Iteration {i+1}: {inference_time:.4f}s ({current_fps:.1f} FPS)")

        # Calculate statistics
        avg_time = np.mean(times)
        std_time = np.std(times)
        min_time = np.min(times)
        max_time = np.max(times)
        avg_fps = 1.0 / avg_time
        max_fps = 1.0 / min_time
        min_fps = 1.0 / max_time

        # Parse detections
        result = results[0]
        detections = []
        adas_objects = ['person', 'bicycle', 'car', 'motorcycle', 'bus', 'truck', 'traffic light', 'stop sign']

        if result.boxes is not None:
            for box in result.boxes:
                class_id = int(box.cls[0])
                class_name = result.names[class_id]
                confidence = float(box.conf[0])

                if class_name in adas_objects:
                    detections.append({
                        'class': class_name,
                        'confidence': confidence
                    })

        return {
            'avg_fps': avg_fps,
            'min_fps': min_fps,
            'max_fps': max_fps,
            'avg_time': avg_time,
            'std_time': std_time,
            'fps_std': std_time * avg_fps * avg_fps,
            'all_times': times,
            'objects': len(detections),
            'detections': detections
        }

    def detect_lanes(self, image_path):
        """Simple lane detection"""
        image = cv2.imread(image_path)
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

        # Optimized preprocessing
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        edges = cv2.Canny(blur, 50, 150)

        # ROI
        height, width = gray.shape
        mask = np.zeros_like(edges)
        polygon = np.array([[
            (width//4, height),
            (width//2 - 50, height//2 + 50),
            (width//2 + 50, height//2 + 50),
            (3*width//4, height)
        ]], np.int32)
        cv2.fillPoly(mask, polygon, 255)
        masked_edges = cv2.bitwise_and(edges, mask)

        # Line detection
        lines = cv2.HoughLinesP(masked_edges, 1, np.pi/180, 50,
                               minLineLength=100, maxLineGap=50)

        return len(lines) if lines is not None else 0

    def compare_performance(self, image_path):
        """Compare PyTorch vs TensorRT performance"""
        print(f"\n🔥 Performance Comparison: {os.path.basename(image_path)}")
        print("=" * 60)

        # PyTorch benchmark
        pytorch_results = self.benchmark_inference(image_path, 'pytorch', 20)

        # TensorRT benchmark
        tensorrt_results = self.benchmark_inference(image_path, 'tensorrt', 20)

        # Lane detection (same for both)
        lanes = self.detect_lanes(image_path)

        # Calculate improvements
        fps_improvement = (tensorrt_results['avg_fps'] / pytorch_results['avg_fps'] - 1) * 100
        time_improvement = (pytorch_results['avg_time'] / tensorrt_results['avg_time'] - 1) * 100

        print(f"\n📊 COMPARISON RESULTS:")
        print(f"  PyTorch FPS: {pytorch_results['avg_fps']:.2f} (±{pytorch_results['fps_std']:.2f})")
        print(f"  TensorRT FPS: {tensorrt_results['avg_fps']:.2f} (±{tensorrt_results['fps_std']:.2f})")
        print(f"  🚀 FPS Improvement: {fps_improvement:.1f}%")
        print(f"  ⚡ Time Reduction: {time_improvement:.1f}%")
        print(f"  🎯 Objects: PT={pytorch_results['objects']}, RT={tensorrt_results['objects']}")
        print(f"  🛣️ Lanes: {lanes}")

        return {
            'image_name': os.path.basename(image_path),
            'pytorch': pytorch_results,
            'tensorrt': tensorrt_results,
            'lanes': lanes,
            'fps_improvement': fps_improvement,
            'time_improvement': time_improvement
        }

def create_tensorrt_comparison_charts():
    """Create TensorRT vs PyTorch comparison charts"""
    print("🚀 Creating TensorRT vs PyTorch Comparison Charts...")

    # Check available images
    image_files = ['1.jpg', '2.jpg', '3.jpg']
    available_images = []

    for img_file in image_files:
        full_path = f'/workspace/{img_file}'
        if os.path.exists(full_path):
            available_images.append(full_path)

    if not available_images:
        print("❌ No images found!")
        return

    # Initialize TensorRT ADAS
    adas = TensorRTADAS()

    # Compare all images
    results = []
    for image_path in available_images:
        result = adas.compare_performance(image_path)
        results.append(result)

    # Create comparison charts
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    fig.suptitle('TensorRT vs PyTorch ADAS Performance Comparison', fontsize=16, fontweight='bold')

    # Extract data
    image_names = [r['image_name'] for r in results]
    pytorch_fps = [r['pytorch']['avg_fps'] for r in results]
    tensorrt_fps = [r['tensorrt']['avg_fps'] for r in results]
    fps_improvements = [r['fps_improvement'] for r in results]
    pytorch_objects = [r['pytorch']['objects'] for r in results]
    tensorrt_objects = [r['tensorrt']['objects'] for r in results]
    lanes = [r['lanes'] for r in results]

    # Colors
    pytorch_color = '#FF6B6B'
    tensorrt_color = '#4ECDC4'
    improvement_color = '#45B7D1'

    # 1. FPS Comparison
    x = np.arange(len(image_names))
    width = 0.35

    bars1 = axes[0, 0].bar(x - width/2, pytorch_fps, width, label='PyTorch',
                          color=pytorch_color, alpha=0.8)
    bars2 = axes[0, 0].bar(x + width/2, tensorrt_fps, width, label='TensorRT',
                          color=tensorrt_color, alpha=0.8)

    axes[0, 0].set_title('FPS Performance Comparison', fontweight='bold')
    axes[0, 0].set_ylabel('Frames Per Second')
    axes[0, 0].set_xticks(x)
    axes[0, 0].set_xticklabels(image_names)
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Add value labels
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            axes[0, 0].text(bar.get_x() + bar.get_width()/2., height + 0.5,
                           f'{height:.1f}', ha='center', va='bottom', fontweight='bold')

    # 2. Performance Improvement
    bars = axes[0, 1].bar(image_names, fps_improvements, color=improvement_color, alpha=0.8)
    axes[0, 1].set_title('FPS Improvement (%)', fontweight='bold')
    axes[0, 1].set_ylabel('Improvement Percentage')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].axhline(y=0, color='black', linestyle='-', alpha=0.5)

    # Add value labels
    for bar in bars:
        height = bar.get_height()
        axes[0, 1].text(bar.get_x() + bar.get_width()/2., height + 1,
                       f'{height:.1f}%', ha='center', va='bottom', fontweight='bold')

    # 3. Object Detection Accuracy
    bars1 = axes[0, 2].bar(x - width/2, pytorch_objects, width, label='PyTorch',
                          color=pytorch_color, alpha=0.8)
    bars2 = axes[0, 2].bar(x + width/2, tensorrt_objects, width, label='TensorRT',
                          color=tensorrt_color, alpha=0.8)

    axes[0, 2].set_title('Objects Detected', fontweight='bold')
    axes[0, 2].set_ylabel('Number of Objects')
    axes[0, 2].set_xticks(x)
    axes[0, 2].set_xticklabels(image_names)
    axes[0, 2].legend()
    axes[0, 2].grid(True, alpha=0.3)

    # Add value labels
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            axes[0, 2].text(bar.get_x() + bar.get_width()/2., height + 0.1,
                           f'{int(height)}', ha='center', va='bottom', fontweight='bold')

    # 4. Processing Time Comparison
    pytorch_times = [r['pytorch']['avg_time'] * 1000 for r in results]  # Convert to ms
    tensorrt_times = [r['tensorrt']['avg_time'] * 1000 for r in results]

    bars1 = axes[1, 0].bar(x - width/2, pytorch_times, width, label='PyTorch',
                          color=pytorch_color, alpha=0.8)
    bars2 = axes[1, 0].bar(x + width/2, tensorrt_times, width, label='TensorRT',
                          color=tensorrt_color, alpha=0.8)

    axes[1, 0].set_title('Processing Time Comparison', fontweight='bold')
    axes[1, 0].set_ylabel('Time (milliseconds)')
    axes[1, 0].set_xticks(x)
    axes[1, 0].set_xticklabels(image_names)
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Add value labels
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            axes[1, 0].text(bar.get_x() + bar.get_width()/2., height + 1,
                           f'{height:.1f}', ha='center', va='bottom', fontweight='bold')

    # 5. Performance Summary
    axes[1, 1].axis('off')

    avg_pytorch_fps = np.mean(pytorch_fps)
    avg_tensorrt_fps = np.mean(tensorrt_fps)
    avg_improvement = np.mean(fps_improvements)

    summary_text = f"""
TENSORRT OPTIMIZATION RESULTS
{'='*35}

Average Performance:
• PyTorch FPS: {avg_pytorch_fps:.1f}
• TensorRT FPS: {avg_tensorrt_fps:.1f}
• Average Improvement: {avg_improvement:.1f}%

Best Performance:
• Highest FPS: {max(tensorrt_fps):.1f} ({image_names[np.argmax(tensorrt_fps)]})
• Best Improvement: {max(fps_improvements):.1f}% ({image_names[np.argmax(fps_improvements)]})

Detection Accuracy:
• PyTorch Objects: {sum(pytorch_objects)}
• TensorRT Objects: {sum(tensorrt_objects)}
• Accuracy Maintained: {'✅' if sum(pytorch_objects) == sum(tensorrt_objects) else '⚠️'}

Lane Detection:
• Total Lanes: {sum(lanes)}
• Avg per Image: {np.mean(lanes):.1f}
"""

    axes[1, 1].text(0.05, 0.95, summary_text, transform=axes[1, 1].transAxes,
                    fontsize=10, verticalalignment='top', fontfamily='monospace',
                    bbox=dict(boxstyle="round,pad=0.5", facecolor="lightgreen", alpha=0.8))

    # 6. Speed vs Accuracy Trade-off
    axes[1, 2].scatter(pytorch_fps, pytorch_objects, s=100, color=pytorch_color,
                      alpha=0.8, label='PyTorch', marker='o')
    axes[1, 2].scatter(tensorrt_fps, tensorrt_objects, s=100, color=tensorrt_color,
                      alpha=0.8, label='TensorRT', marker='s')

    # Add image labels
    for i, name in enumerate(image_names):
        axes[1, 2].annotate(name, (pytorch_fps[i], pytorch_objects[i]),
                           xytext=(5, 5), textcoords='offset points', fontsize=8)
        axes[1, 2].annotate(name, (tensorrt_fps[i], tensorrt_objects[i]),
                           xytext=(5, -10), textcoords='offset points', fontsize=8)

    axes[1, 2].set_title('Speed vs Accuracy Trade-off', fontweight='bold')
    axes[1, 2].set_xlabel('FPS (Speed)')
    axes[1, 2].set_ylabel('Objects Detected (Accuracy)')
    axes[1, 2].legend()
    axes[1, 2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()  # 이 줄을 명시적으로 추가

    # Print detailed comparison
    print(f"\n🏆 TENSORRT OPTIMIZATION SUMMARY")
    print("="*60)

    for i, result in enumerate(results):
        print(f"\n{i+1}. {result['image_name']}:")
        print(f"   PyTorch:  {result['pytorch']['avg_fps']:.2f} FPS ({result['pytorch']['avg_time']*1000:.1f}ms)")
        print(f"   TensorRT: {result['tensorrt']['avg_fps']:.2f} FPS ({result['tensorrt']['avg_time']*1000:.1f}ms)")
        print(f"   🚀 Improvement: {result['fps_improvement']:.1f}% faster")
        print(f"   🎯 Objects: {result['pytorch']['objects']} → {result['tensorrt']['objects']}")
        print(f"   🛣️ Lanes: {result['lanes']}")

    print(f"\n🎉 OVERALL OPTIMIZATION RESULTS:")
    print(f"   Average FPS improvement: {avg_improvement:.1f}%")
    print(f"   Best single improvement: {max(fps_improvements):.1f}%")
    print(f"   Accuracy maintained: {'Yes' if sum(pytorch_objects) == sum(tensorrt_objects) else 'No'}")

    return results

def create_visual_comparison():
    """Create visual comparison showing original vs processed images"""
    print("\n🖼️ Creating Visual Comparison Results...")

    # Check available images
    image_files = ['1.jpg', '2.jpg', '3.jpg']
    available_images = []

    for img_file in image_files:
        full_path = f'/workspace/{img_file}'
        if os.path.exists(full_path):
            available_images.append(full_path)

    if not available_images:
        return

    # Initialize ADAS system
    adas = TensorRTADAS()

    # Create visual comparison figure
    fig, axes = plt.subplots(len(available_images), 3, figsize=(20, 7*len(available_images)))
    if len(available_images) == 1:
        axes = axes.reshape(1, -1)

    fig.suptitle('ADAS Visual Results: Original → PyTorch → TensorRT',
                 fontsize=18, fontweight='bold', y=0.98)

    for i, image_path in enumerate(available_images):
        print(f"📸 Processing visual results for: {os.path.basename(image_path)}")

        # Quick performance test for display
        pytorch_result = adas.benchmark_inference(image_path, 'pytorch', 5)
        tensorrt_result = adas.benchmark_inference(image_path, 'tensorrt', 5)
        lanes = adas.detect_lanes(image_path)

        # Load original image
        original = cv2.imread(image_path)
        original_rgb = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)

        # Get PyTorch results with visualization
        pytorch_yolo_results = adas.model(image_path, conf=0.5, verbose=False)
        pytorch_annotated = pytorch_yolo_results[0].plot()
        pytorch_annotated_rgb = cv2.cvtColor(pytorch_annotated, cv2.COLOR_BGR2RGB)

        # Get TensorRT results with visualization
        tensorrt_yolo_results = adas.tensorrt_model(image_path, conf=0.5, verbose=False)
        tensorrt_annotated = tensorrt_yolo_results[0].plot()
        tensorrt_annotated_rgb = cv2.cvtColor(tensorrt_annotated, cv2.COLOR_BGR2RGB)

        # Add lane detection to both processed images
        height, width = pytorch_annotated_rgb.shape[:2]

        # Add lane lines to PyTorch result
        pytorch_with_lanes = pytorch_annotated_rgb.copy()
        lane_image = cv2.imread(image_path)
        gray = cv2.cvtColor(lane_image, cv2.COLOR_BGR2GRAY)
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        edges = cv2.Canny(blur, 50, 150)

        # ROI and lane detection
        mask = np.zeros_like(edges)
        polygon = np.array([[
            (width//4, height), (width//2 - 50, height//2 + 50),
            (width//2 + 50, height//2 + 50), (3*width//4, height)
        ]], np.int32)
        cv2.fillPoly(mask, polygon, 255)
        masked_edges = cv2.bitwise_and(edges, mask)
        lines = cv2.HoughLinesP(masked_edges, 1, np.pi/180, 50,
                               minLineLength=100, maxLineGap=50)

        # Draw lanes on both results
        if lines is not None:
            for line in lines:
                x1, y1, x2, y2 = line[0]
                cv2.line(pytorch_with_lanes, (x1, y1), (x2, y2), (0, 255, 255), 3)

        tensorrt_with_lanes = tensorrt_annotated_rgb.copy()
        if lines is not None:
            for line in lines:
                x1, y1, x2, y2 = line[0]
                cv2.line(tensorrt_with_lanes, (x1, y1), (x2, y2), (0, 255, 255), 3)

        # Add performance information
        # PyTorch info
        cv2.putText(pytorch_with_lanes, f"PyTorch ADAS", (10, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 3)
        cv2.putText(pytorch_with_lanes, f"FPS: {pytorch_result['avg_fps']:.1f}", (10, 70),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.putText(pytorch_with_lanes, f"Objects: {pytorch_result['objects']}", (10, 110),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.putText(pytorch_with_lanes, f"Lanes: {lanes}", (10, 150),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

        # TensorRT info
        improvement = (tensorrt_result['avg_fps'] / pytorch_result['avg_fps'] - 1) * 100
        cv2.putText(tensorrt_with_lanes, f"TensorRT ADAS", (10, 30),
                   cv2.FONT_HERSHEY_SIMPLEX, 1, (255, 255, 255), 3)
        cv2.putText(tensorrt_with_lanes, f"FPS: {tensorrt_result['avg_fps']:.1f}", (10, 70),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.putText(tensorrt_with_lanes, f"Objects: {tensorrt_result['objects']}", (10, 110),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.putText(tensorrt_with_lanes, f"Improvement: +{improvement:.1f}%", (10, 150),
                   cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

        # Display images
        # Original
        axes[i, 0].imshow(original_rgb)
        axes[i, 0].set_title(f"Original: {os.path.basename(image_path)}", fontsize=14, fontweight='bold')
        axes[i, 0].axis('off')

        # PyTorch result
        axes[i, 1].imshow(pytorch_with_lanes)
        title_pt = f"PyTorch Result\nFPS: {pytorch_result['avg_fps']:.1f}, Objects: {pytorch_result['objects']}, Lanes: {lanes}"
        axes[i, 1].set_title(title_pt, fontsize=12, fontweight='bold')
        axes[i, 1].axis('off')

        # TensorRT result
        axes[i, 2].imshow(tensorrt_with_lanes)
        title_rt = f"TensorRT Result\nFPS: {tensorrt_result['avg_fps']:.1f} (+{improvement:.1f}%), Objects: {tensorrt_result['objects']}"
        axes[i, 2].set_title(title_rt, fontsize=12, fontweight='bold', color='red')
        axes[i, 2].axis('off')

        # Add comparison arrows and text
        if len(available_images) == 1:
            # Add arrows between images
            fig.text(0.31, 0.5, '→', fontsize=40, ha='center', va='center', color='blue', weight='bold')
            fig.text(0.64, 0.5, '→', fontsize=40, ha='center', va='center', color='red', weight='bold')
            fig.text(0.31, 0.45, 'YOLO\nDetection', fontsize=10, ha='center', va='center', color='blue', weight='bold')
            fig.text(0.64, 0.45, 'TensorRT\nOptimization', fontsize=10, ha='center', va='center', color='red', weight='bold')

        print(f"  ✅ PyTorch: {pytorch_result['avg_fps']:.1f} FPS, {pytorch_result['objects']} objects")
        print(f"  🚀 TensorRT: {tensorrt_result['avg_fps']:.1f} FPS, {tensorrt_result['objects']} objects")
        print(f"  📈 Improvement: +{improvement:.1f}%")

    plt.tight_layout()
    plt.subplots_adjust(top=0.94)
    plt.show()  # 시각적 비교 차트도 명시적으로 표시

    # Print visual comparison summary
    print(f"\n🎨 VISUAL COMPARISON COMPLETE")
    print("="*50)
    print(f"📸 Images processed: {len(available_images)}")
    print(f"🖼️ Comparison format: Original → PyTorch → TensorRT")
    print(f"✅ Lane detection overlaid on both results")
    print(f"📊 Performance metrics displayed on images")

# Main execution with visual comparison
def main():
    """Main execution for TensorRT comparison with visual results"""
    print("🚀💨 Starting TensorRT vs PyTorch ADAS Comparison")
    print("="*60)

    # Create TensorRT performance comparison charts
    print("📊 Creating performance comparison charts...")
    results = create_tensorrt_comparison_charts()

    # Force display charts if not showing
    plt.ion()  # Turn on interactive mode

    # Create visual comparison
    print("\n🖼️ Creating visual comparison...")
    create_visual_comparison()

    # Make sure all plots are displayed
    plt.show(block=False)
    plt.draw()

    print("\n✅ Complete TensorRT optimization analysis finished!")
    print("🏆 Performance improvements measured and visualized!")
    print("🖼️ Visual comparison of original vs processed images completed!")
    print("📊 If charts are not visible, try running plt.show() manually")

    return results

if __name__ == "__main__":
    tensorrt_results = main()